# Deployment Setup
### Purpose:
- Install dependencies
- Mount Google Drive
- Prepare deployment environment

In [ ]:
# Install required packages
!pip install -q gradio langchain langchain-groq langchain-community langchain-huggingface faiss-cpu

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Setup Complete!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Mounted at /content/drive
✅ Setup Complete!


# Import Libraries
### Purpose:
- Load Gradio
- Load LangChain
- Load FAISS
- Load embedding models

In [ ]:
import os
import json
import gradio as gr
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/tmp/ipykernel_3791/2650721859.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


# Complete Multi-Agent Backend System
### Contains:
- Agent 1: Symptom Extraction
- Agent 2: Medical Context Retrieval (RAG)
- Agent 3: Disease Prediction
- Agent 4: Medication Guidance
- Agent 5: Recommendation & Safety

In [ ]:
## CONFIG
os.environ["GROQ_API_KEY"] = "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"  # Your key

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.1)

# ====================== AGENT 1: Symptom Extraction ======================
def extract_symptoms(user_input):
    prompt = PromptTemplate.from_template("""
You are an expert medical NLP assistant.
Extract symptoms from the user message.
Return ONLY valid JSON in this exact format. No extra text.

{{
  "symptoms": [
    {{
      "symptom": "headache",
      "severity": "severe",
      "duration": "3 days",
      "body_part": "head"
    }}
  ]
}}

User Input: {user_input}
""")

    chain = prompt | llm
    response = chain.invoke({"user_input": user_input})
    cleaned = response.content.strip().replace("```json", "").replace("```", "").strip()

    try:
        return json.loads(cleaned)
    except:
        return {"error": "Failed to parse", "raw": response.content}
# ====================== AGENT 2: RAG Retrieval ======================
def retrive_medical_context(symptoms_dict, k=5):
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    # IMPORTANT: Update path if needed
    index_path = "/content/drive/MyDrive/symptom-checker/Symptor_checker_dnlp_project/medical_faiss_index"

    vectorstore = FAISS.load_local(index_path, embeddings, allow_dangerous_deserialization=True)

    query = json.dumps(symptoms_dict)
    docs = vectorstore.similarity_search(query, k=k)

    context = ""
    sources = []
    for i, doc in enumerate(docs):
        context += f"\n--- Document {i+1} ---\n{doc.page_content}\n"
        sources.append(doc.metadata.get('source', 'Unknown'))

    return {"context": context.strip(), "sources": sources, "num_docs": len(docs)}
# ====================== AGENT 3: Disease Prediction ======================
def predict_disease(symptoms_dict, retrieved_context):
    prompt = PromptTemplate.from_template("""
You are a medical assistant. Analyze the symptoms and context.
Only use information from the retrieved context.
List 2-4 possible conditions with brief reasons.
Use words like "may suggest".

Symptoms: {symptoms}
Context: {context}

Return in simple bullet points.
""")
    chain = prompt | llm
    response = chain.invoke({
        "symptoms": json.dumps(symptoms_dict, indent=2),
        "context": retrieved_context["context"]
    })
    return response.content
# ====================== HELPER: Severity ======================
def classify_severity(symptoms_dict):
    text = json.dumps(symptoms_dict).lower()
    if any(kw in text for kw in ["chest pain", "difficulty breathing", "unconscious", "severe bleeding", "stroke"]):
        return "EMERGENCY"
    if any(kw in text for kw in ["severe", "intense", "worst"]):
        return "HIGH"
    if "moderate" in text or "fever" in text:
        return "MODERATE"
    return "MILD"
# ====================== AGENT 4: Medication Guidance ======================
def get_medication_guidance(symptoms_dict, prediction, severity):
    prompt = PromptTemplate.from_template("""
You are a safe medication assistant.
Suggest ONLY common OTC medicines or home remedies.
Never suggest antibiotics or prescription drugs.

Symptoms: {symptoms}
Prediction: {prediction}
Severity: {severity}

Output format:
- Home Remedies: ...
- Suggested OTC: ...
- Recommendation: ...
- Disclaimer: ...
""")
    chain = prompt | llm
    response = chain.invoke({
        "symptoms": json.dumps(symptoms_dict),
        "prediction": prediction,
        "severity": severity
    })
    return response.content
# ====================== AGENT 5: Final Recommendations ======================
def generate_recommendations(symptoms_dict, prediction, medication, severity):
    prompt = PromptTemplate.from_template("""
Summarize into clear advice:

Symptoms: {symptoms}
Conditions: {prediction}
Medication: {medication}
Severity: {severity}

Output in bullet points:
- Overall Assessment
- Action Plan
- When to Seek Emergency Help
- Lifestyle Tips
- Final Disclaimer
""")
    chain = prompt | llm
    response = chain.invoke({
        "symptoms": json.dumps(symptoms_dict),
        "prediction": prediction,
        "medication": medication,
        "severity": severity
    })
    return response.content
# ====================== IMPROVED FULL PIPELINE (Better UI) ======================
def run_symptom_pipeline(user_input):
    try:
        if not user_input or len(user_input.strip()) < 10:
            # Returning error formats for all structural outputs
            return "⚠️ Input too short", "", "❌ Please enter a detailed description of your symptoms.", "", ""

        symptoms = extract_symptoms(user_input)
        retrieved = retrive_medical_context(symptoms)
        prediction = predict_disease(symptoms, retrieved)
        severity = classify_severity(symptoms)
        medication = get_medication_guidance(symptoms, prediction, severity)
        recommendations = generate_recommendations(symptoms, prediction, medication, severity)

        # 1. Format Symptoms Column
        symptoms_html = ""
        if isinstance(symptoms, dict) and "symptoms" in symptoms:
            for sym in symptoms.get("symptoms", []):
                symptoms_html += f"• **{sym.get('symptom','Unknown').title()}**<br>  ↳ *Severity:* `{sym.get('severity','N/A')}` | *Duration:* `{sym.get('duration','N/A')}` | *Body Part:* `{sym.get('body_part','N/A')}`\n\n"
        else:
            symptoms_html = "Could not structure symptoms."

        # 2. Format Severity Colored Banner
        color_map = {"EMERGENCY": "#ff4d4d", "HIGH": "#ff8533", "MODERATE": "#ffcc00", "MILD": "#2db300"}
        text_color = "#ffffff" if severity in ["EMERGENCY", "HIGH"] else "#000000"
        severity_badge = f"""
        <div style="background-color: {color_map.get(severity, '#eceff1')}; color: {text_color}; padding: 15px; border-radius: 8px; text-align: center; font-weight: bold; font-size: 1.3em;">
            ⚠️ SEVERITY LEVEL: {severity}
        </div>
        """

        # 3. Compile full markdown report for the summary tab
        full_report = f"""
### 📊 Complete Case Profile
**Original Query:** *"{user_input}"*

**Sources Consulted:** `dataset.csv (Medical Knowledge Base RAG)`

---
{recommendations}

---
*⚠️ **Disclaimer**: This is an educational academic project. Not a substitute for professional medical advice.*
"""

        return severity_badge, symptoms_html, prediction, medication, full_report

    except Exception as e:
        error_msg = f"❌ **Error**: {str(e)}"
        return "⚠️ Error", error_msg, error_msg, error_msg, error_msg


# Gradio User Interface (Frontend)
### Purpose:
- Create patient interface
- Connect UI with backend agents
- Display predictions and recommendations

In [ ]:
def create_ui():
    # Using a clean, professional layout theme
    with gr.Blocks(title="AI Symptom Checker", theme=gr.themes.Soft()) as demo:

        # Header banner with an illustrative clinical look
        gr.Markdown(
            """
            # 🏥 **DNLP - Multi-Agent Clinical AI Assistant**
            ### *Advanced Natural Language Processing Semester Project*
            ---
            """
        )

        with gr.Row():
            # Left Input Column
            with gr.Column(scale=2):
                gr.Markdown("### 📥 **Patient Intake**")
                input_text = gr.Textbox(
                    label="Describe your current symptoms in detail:",
                    placeholder="Example: I've had a sharp stomach ache and mild nausea since last night after dinner.",
                    lines=6
                )

                submit_btn = gr.Button("🔍 Run AI Diagnostics Pipeline", variant="primary", size="lg")

                # Interactive Quick Examples
                gr.Examples(
                    examples=[
                        ["I have a severe headache and high fever for 3 days with intense body pain."],
                        ["Persistent cough and sore throat for the last 4 days with mild fever."],
                        ["Chest pain and difficulty breathing since morning."]
                    ],
                    inputs=input_text,
                    label="💡 Click a sample case scenario to test:"
                )

            # Right Output Column (Hidden initially until submission)
            with gr.Column(scale=3):
                gr.Markdown("### 📤 **Analysis & Diagnostic Insights**")

                # 1. Severity Alert Banner Display
                severity_output = gr.HTML("<div style='text-align:center; color:#777;'>Submit symptoms to see alert status</div>")

                # Tabbed Interface to break up text overload
                with gr.Tabs():
                    with gr.TabItem("⚕️ Extracted Profile & Conditions"):
                        with gr.Row():
                            with gr.Column():
                                gr.Markdown("#### **Extracted Symptoms**")
                                symptoms_output = gr.Markdown("_No data collected yet_")
                            with gr.Column():
                                gr.Markdown("#### **Differential Predictions**")
                                prediction_output = gr.Markdown("_Awaiting pipeline analysis..._")

                    with gr.TabItem("💊 Clinical Care Guide"):
                        gr.Markdown("#### **Home Remediation & OTC Assistance**")
                        medication_output = gr.Markdown("_Awaiting pipeline analysis..._")

                    with gr.TabItem("📋 Full Patient Summary"):
                        summary_output = gr.Markdown("_Complete generated report will show here._")

        # Footer
        gr.Markdown(
            """
            ---
            *Built with LangChain • Groq Llama-3.3 • SentenceTransformers • FAISS Vector DB • Gradio Interface*
            """
        )

        # Wire up the button mapping variables properly
        submit_btn.click(
            fn=run_symptom_pipeline,
            inputs=input_text,
            outputs=[severity_output, symptoms_output, prediction_output, medication_output, summary_output]
        )

    return demo

# Launch the UI
demo = create_ui()
demo.launch(share=True, debug=True)

/tmp/ipykernel_3791/92365336.py:3: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="AI Symptom Checker", theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://870ed6d614b62b079d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://870ed6d614b62b079d.gradio.live
